<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/02_pca_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U2 主成分分析(PCA)(notebook)

本文:[`02-pca.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/unsupervised/02-pca.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`unsupervised/02_pca.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

# 相関のある 3 次元のデータ:本当は 2 次元の構造(latent)を、3 次元に埋め込んで雑音を足した
rng = np.random.default_rng(0)
latent = rng.normal(0, [3.0, 1.0], (300, 2))
A_embed = np.array([[1.0, 0.2], [0.5, 1.0], [0.3, -0.8]])
X = latent @ A_embed.T + rng.normal(0, 0.3, (300, 3))
print("X の形:", X.shape)

## 1. PCA を実装する

本文 §2 の3つの手順を実装する。`np.linalg.eigh` は固有値を**小さい順**に返すので、大きい順に並べ替える(固有ベクトルの列も同じ順に)。

**手計算**:本文の確認課題1を解く。

- 手計算:(ここに書く)

In [ ]:
def pca_fit(X):
    mean = X.mean(axis=0)
    Xc = X - mean
    S = Xc.T @ Xc / len(X)
    eigvals, eigvecs = np.linalg.eigh(S)
    order = np.argsort(eigvals)[::-1]                        # 大きい順
    return eigvals[order], eigvecs[:, order], mean

In [ ]:
X_hand = np.array([[1.0, 1.0], [-1.0, -1.0], [2.0, 2.0], [-2.0, -2.0]])
ev_h, U_h, _ = pca_fit(X_hand)
check("確認課題1:固有値は (5, 0)", ev_h, [5.0, 0.0], tol=1e-9)
check("確認課題1:第1主成分は (1, 1)/sqrt(2) の向き(符号は問わない)", np.abs(U_h[:, 0]), [1 / np.sqrt(2)] * 2)

eigvals, U, mean = pca_fit(X)
ratio = eigvals / eigvals.sum()
print("固有値:", eigvals.round(3))
print("寄与率:", ratio.round(3), "  累積寄与率:", np.cumsum(ratio).round(3))
check("固有値の和 = 各列の分散の和", eigvals.sum(), X.var(axis=0).sum())
check("固有ベクトルは互いに直交し、長さ 1(U^T U = I)", U.T @ U, np.eye(3))
check("第1主成分の方向の分散 = 最大の固有値(式 (1))", np.var((X - mean) @ U[:, 0]), eigvals[0])

## 2. SVD で確かめる

本文 §5 のとおり、中心化した $X_c$ の SVD の $V$ の列が主成分、 $\sigma_j^2 / N$ が固有値になることを確かめる。

In [ ]:
Xc = X - mean
_, s, Vt = np.linalg.svd(Xc, full_matrices=False)
check("固有値 = 特異値の2乗 / N", eigvals, s ** 2 / len(X))
check("主成分 = V の列(符号は問わないので絶対値で比べる)", np.abs(U), np.abs(Vt.T))

## 3. 射影と再構成

本文の式 (3) を実装し、再構成の誤差が、捨てた固有値の和(式 (4))と一致することを確かめる。

**予想**:$M = 1, 2, 3$ で、再構成の誤差はそれぞれいくつになるか(上の固有値から計算する)。

- 予想:(ここに書く)

In [ ]:
def pca_transform(X, U, mean, M):
    return (X - mean) @ U[:, :M]

def pca_inverse(Z, U, mean):
    M = Z.shape[1]
    return Z @ U[:, :M].T + mean

In [ ]:
for M in [1, 2, 3]:
    X_rec = pca_inverse(pca_transform(X, U, mean, M), U, mean)
    err = np.mean(((X - X_rec) ** 2).sum(axis=1))
    check(f"M = {M}:再構成の誤差 {err:.4f} = 捨てた固有値の和", err, eigvals[M:].sum())

Z2 = pca_transform(X, U, mean, 2)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(Z2[:, 0], Z2[:, 1], s=8); ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); ax[0].set_aspect("equal"); ax[0].set_title("projected to 2D")
ax[1].scatter(latent[:, 0], latent[:, 1], s=8, c="gray"); ax[1].set_aspect("equal"); ax[1].set_title("true latent (unknown in practice)")
plt.tight_layout(); plt.show()

主成分得点の図は、本当の2次元の構造(右)を、回転や反転を除いて、ほぼ再現している。

## 4. 前処理:単位がそろっていないとき

3列目だけ単位を変えて、値を 100 倍にする(m と cm のように)。標準化しないと、何が起きるか(本文 §6)。

**予想**:第1主成分の寄与率と向きはどうなるか。

- 予想:(ここに書く)

In [ ]:
X_bad = X.copy(); X_bad[:, 2] *= 100
ev_bad, U_bad, _ = pca_fit(X_bad)
print("100 倍した列があると:寄与率", (ev_bad / ev_bad.sum()).round(4), "  第1主成分", U_bad[:, 0].round(3))
X_std = (X_bad - X_bad.mean(axis=0)) / X_bad.std(axis=0)
ev_std, U_std, _ = pca_fit(X_std)
print("標準化してから:        寄与率", (ev_std / ev_std.sum()).round(4), "  第1主成分", U_std[:, 0].round(3))

## 5. 条件を変える

1. 雑音の大きさを 0.3 から 0.05、1.0、3.0 に変えると、累積寄与率 90% に必要な次元はどう変わるか
2. 10 次元のデータ(本当は 2 次元の構造)で、寄与率を棒グラフで描く

In [ ]:
def n_components_for(ratio_target, eigvals):
    return int(np.searchsorted(np.cumsum(eigvals) / eigvals.sum(), ratio_target) + 1)

for noise in [0.05, 0.3, 1.0, 3.0]:
    Xn = latent @ A_embed.T + np.random.default_rng(1).normal(0, noise, (300, 3))
    print(f"1. 雑音 {noise:4}:累積寄与率 90% に必要な次元 {n_components_for(0.9, pca_fit(Xn)[0])}")

A10 = np.random.default_rng(2).normal(size=(10, 2))
X10 = latent @ A10.T + np.random.default_rng(3).normal(0, 0.3, (300, 10))
ev10 = pca_fit(X10)[0]
plt.bar(range(1, 11), ev10 / ev10.sum()); plt.xlabel("component"); plt.ylabel("explained variance ratio"); plt.title("10-D data with 2-D structure"); plt.show()

## 6. 研究で使うときは

`cpai_lab` には PCA の関数は入っていない(本文 §8)。scikit-learn の `PCA` との対応は次のとおり。

| この notebook | scikit-learn |
|---|---|
| `pca_fit` の固有値 | `PCA().fit(X).explained_variance_`(ただし $N - 1$ で割った値) |
| 寄与率 | `explained_variance_ratio_` |
| 固有ベクトル `U` の列 | `components_` の行 |
| `pca_transform`、`pca_inverse` | `transform`、`inverse_transform` |

## 7. 説明(自分の言葉で書く)

1. 主成分が共分散行列の固有ベクトルになるのはなぜか
   - (ここに書く)
2. 寄与率は何を表しているか。再構成の誤差とどう関係するか
   - (ここに書く)
3. PCA の前に標準化が必要になるのは、どんなときか
   - (ここに書く)